# Part 2 — Step 8: Domain Adaptation

Bridges the gap between ACNE04 (face selfies) and DermNet (clinical photos) through a staged pipeline:

| Step | Technique |
|---|---|
| 1 | Color normalization — Reinhard LAB + histogram matching |
| 2 | Training augmentation — ColorJitter, RandomResizedCrop, GaussianBlur |
| 3 | 2-stage few-shot fine-tuning — 20 DermNet samples, two-stage LR |
| 4 | Threshold optimization — swept 0.05–0.95 on the 20 training samples |
| 5 | 6-crop TTA — FiveCrop + center flip, average 6 forward passes |
| 6 | FaceNet VGGFace2 — face-pretrained backbone as alternative |

**Prerequisites:** Run `06_patch_extraction.ipynb` and `07_train_classifier.ipynb` first.

**Output:** `outputs/classifier/finetuned.pth`, `outputs/part2/dermnet_results.json`, `outputs/part2/final_model_results.png`

In [ ]:
import os
from pathlib import Path

if Path('/content').exists():
    os.chdir('/content/AcneDetection')
print(f'Working directory: {os.getcwd()}')

In [ ]:
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    roc_curve, confusion_matrix, ConfusionMatrixDisplay,
)
from PIL import Image
from skimage import color as skcolor
from skimage.exposure import match_histograms

%matplotlib inline

PATCH_DIR   = Path('data/patches')
DERMNET_DIR = Path('data/dermnet')
CLF_OUT     = Path('outputs/classifier')
OUT_PART2   = Path('outputs/part2')
OUT_PART2.mkdir(parents=True, exist_ok=True)
ACNE_FOLDER = 'Acne and Rosacea Photos'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
random.seed(42)

## 1. Domain Gap Visualization

Side-by-side comparison of ACNE04 patches vs DermNet acne images.

In [ ]:
acne04_samples  = random.sample(list((PATCH_DIR / 'train' / 'acne').glob('*.jpg')), 4)
dermnet_samples = random.sample(
    [p for p in (DERMNET_DIR / 'train' / ACNE_FOLDER).glob('*')
     if p.suffix.lower() in ('.jpg', '.jpeg', '.png')], 4)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col, f in enumerate(acne04_samples):
    axes[0][col].imshow(Image.open(f))
    axes[0][col].set_title('ACNE04 patch', fontsize=9)
    axes[0][col].axis('off')
for col, f in enumerate(dermnet_samples):
    axes[1][col].imshow(Image.open(f))
    axes[1][col].set_title('DermNet acne', fontsize=9)
    axes[1][col].axis('off')

plt.suptitle('Domain Gap — ACNE04 (top) vs DermNet (bottom)', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_PART2 / 'domain_gap.png', dpi=150)
plt.show()
print('Saved -> outputs/part2/domain_gap.png')

## 2. RGB Channel Statistics

Quantifies the domain gap by comparing mean RGB values. Large differences confirm why adaptation is needed.

In [ ]:
def channel_stats(paths, n=200):
    sample = random.sample(list(paths), min(n, len(list(paths))))
    means = []
    for p in sample:
        arr = np.array(Image.open(p).convert('RGB').resize((224, 224)), dtype=float)
        means.append(arr.mean(axis=(0, 1)) / 255.0)
    return np.array(means).mean(axis=0), np.array(means).std(axis=0)

acne04_paths  = list((PATCH_DIR / 'train' / 'acne').glob('*.jpg'))
dermnet_paths = [p for p in (DERMNET_DIR / 'train' / ACNE_FOLDER).glob('*')
                 if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]

a_mean, a_std = channel_stats(acne04_paths)
d_mean, d_std = channel_stats(dermnet_paths)

channels = ['R', 'G', 'B']
x = np.arange(3)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - 0.2, a_mean, 0.35, yerr=a_std, label='ACNE04', color='#FF6B6B', capsize=4)
ax.bar(x + 0.2, d_mean, 0.35, yerr=d_std, label='DermNet', color='#4ECDC4', capsize=4)
ax.set_xticks(x); ax.set_xticklabels(channels)
ax.set_ylabel('Mean pixel value (0-1)')
ax.set_title('RGB Channel Statistics — ACNE04 vs DermNet')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_PART2 / 'rgb_channel_stats.png', dpi=150)
plt.show()

print('ACNE04  mean RGB:', a_mean.round(3))
print('DermNet mean RGB:', d_mean.round(3))
print('Difference      :', (d_mean - a_mean).round(3))
print('Saved -> outputs/part2/rgb_channel_stats.png')

## 3. Training Augmentation

Shows how the same ACNE04 patch looks after augmentation — simulating DermNet's lighting/color/scale conditions.

In [ ]:
aug_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
])

src_img = Image.open(acne04_samples[0]).convert('RGB')

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(src_img); axes[0].set_title('Original', fontsize=9); axes[0].axis('off')
for i in range(1, 6):
    axes[i].imshow(aug_tf(src_img))
    axes[i].set_title(f'Aug {i}', fontsize=9); axes[i].axis('off')

plt.suptitle('Training Augmentation — same patch, 5 random draws', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_PART2 / 'augmentation_examples.png', dpi=150)
plt.show()
print('Saved -> outputs/part2/augmentation_examples.png')

## 4. Color Normalization

**Reinhard (2001) LAB-space normalization** shifts DermNet image statistics to match ACNE04's mean/std in LAB space.  
**Histogram matching** then aligns pixel value distributions channel-by-channel against a reference ACNE04 patch.

In [ ]:
# Build ACNE04 reference statistics from 400 patches
sample_paths = random.sample(acne04_paths, min(400, len(acne04_paths)))

pixels = []
for p in sample_paths:
    arr = np.array(Image.open(p).convert('RGB').resize((224, 224))) / 255.0
    pixels.append(skcolor.rgb2lab(arr).reshape(-1, 3))

acne04_pixels   = np.concatenate(pixels, axis=0)
ACNE04_LAB_MEAN = acne04_pixels.mean(axis=0)
ACNE04_LAB_STD  = acne04_pixels.std(axis=0)
ACNE04_REF_IMG  = Image.open(random.choice(acne04_paths)).convert('RGB').resize((224, 224))

def reinhard_normalize(img_pil):
    arr = np.array(img_pil.convert('RGB').resize((224, 224))) / 255.0
    lab = skcolor.rgb2lab(arr)
    src_mean = lab.reshape(-1, 3).mean(axis=0)
    src_std  = lab.reshape(-1, 3).std(axis=0) + 1e-6
    for c in range(3):
        lab[:, :, c] = ((lab[:, :, c] - src_mean[c]) / src_std[c]
                        * ACNE04_LAB_STD[c] + ACNE04_LAB_MEAN[c])
    lab[:, :, 0] = np.clip(lab[:, :, 0], 0, 100)
    lab[:, :, 1] = np.clip(lab[:, :, 1], -128, 127)
    lab[:, :, 2] = np.clip(lab[:, :, 2], -128, 127)
    return Image.fromarray((np.clip(skcolor.lab2rgb(lab), 0, 1) * 255).astype(np.uint8))

def preprocess_dermnet(img_pil):
    img = reinhard_normalize(img_pil)
    arr = np.array(img.convert('RGB').resize((224, 224)))
    matched = match_histograms(arr, np.array(ACNE04_REF_IMG), channel_axis=-1)
    return Image.fromarray(matched.astype(np.uint8))

# 4-panel visualization
sample_d = Image.open(dermnet_samples[0]).convert('RGB')
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(sample_d.resize((224, 224)));  axes[0].set_title('Original DermNet', fontsize=9)
axes[1].imshow(ACNE04_REF_IMG);               axes[1].set_title('ACNE04 reference', fontsize=9)
axes[2].imshow(reinhard_normalize(sample_d)); axes[2].set_title('Reinhard norm', fontsize=9)
axes[3].imshow(preprocess_dermnet(sample_d)); axes[3].set_title('Histogram match', fontsize=9)
for ax in axes: ax.axis('off')
plt.suptitle('Color Normalization Pipeline', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_PART2 / 'color_normalization.png', dpi=150)
plt.show()
print('Saved -> outputs/part2/color_normalization.png')
print('\nACNE04 LAB mean:', ACNE04_LAB_MEAN.round(3))
print('ACNE04 LAB std :', ACNE04_LAB_STD.round(3))

## 5. Baseline Evaluation on DermNet

Load the ACNE04-trained model (no adaptation) and run it on the full DermNet test set.

In [ ]:
class DermNetBinary(Dataset):
    def __init__(self, split, transform):
        self.samples = []
        root = DERMNET_DIR / split
        for folder in sorted(root.iterdir()):
            label = 1 if folder.name == ACNE_FOLDER else 0
            for img_path in sorted(folder.glob('*')):
                if img_path.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                    self.samples.append((img_path, label))
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return self.transform(Image.open(path).convert('RGB')), label

base_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_ds     = DermNetBinary('test', base_tf)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False,
                         num_workers=2, pin_memory=True)

acne_count = sum(1 for _, l in test_ds.samples if l == 1)
naive_acc  = float(1 - acne_count / len(test_ds))
print(f'Test set: {len(test_ds)} images  (acne={acne_count}, non-acne={len(test_ds)-acne_count})')
print(f'Naive baseline accuracy: {naive_acc:.4f}')

In [ ]:
def load_efficientnet(ckpt_path, device):
    m = models.efficientnet_b0(weights=None)
    m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, 2))
    m.load_state_dict(torch.load(str(ckpt_path), map_location=device, weights_only=False))
    return m.to(device).eval()

model = load_efficientnet(CLF_OUT / 'best.pth', device)
print('Loaded best.pth')

all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        probs = torch.softmax(model(imgs.to(device)), dim=1)[:, 1]
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)
all_preds  = (all_probs >= 0.5).astype(int)

baseline_results = {
    'accuracy': float(accuracy_score(all_labels, all_preds)),
    'f1_acne':  float(f1_score(all_labels, all_preds, pos_label=1, zero_division=0)),
    'auroc':    float(roc_auc_score(all_labels, all_probs)),
}
print(f"No adaptation — accuracy={baseline_results['accuracy']:.4f}  "
      f"f1={baseline_results['f1_acne']:.4f}  auroc={baseline_results['auroc']:.4f}")

### 5b. Color Norm Only

Run the same ACNE04 model but preprocess DermNet images through the Reinhard + histogram matching pipeline first.

In [ ]:
class PreprocessedDataset(Dataset):
    def __init__(self, samples, preprocess_fn, transform):
        self.samples, self.preprocess_fn, self.transform = samples, preprocess_fn, transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = self.preprocess_fn(Image.open(path).convert('RGB'))
        return self.transform(img), label

norm_ds     = PreprocessedDataset(test_ds.samples, preprocess_dermnet, base_tf)
norm_loader = DataLoader(norm_ds, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

norm_probs, norm_labels = [], []
with torch.no_grad():
    for imgs, labels in norm_loader:
        probs = torch.softmax(model(imgs.to(device)), dim=1)[:, 1]
        norm_probs.extend(probs.cpu().numpy())
        norm_labels.extend(labels.numpy())

norm_probs  = np.array(norm_probs)
norm_labels = np.array(norm_labels)
norm_preds  = (norm_probs >= 0.5).astype(int)

color_norm_results = {
    'accuracy': float(accuracy_score(norm_labels, norm_preds)),
    'f1_acne':  float(f1_score(norm_labels, norm_preds, pos_label=1, zero_division=0)),
    'auroc':    float(roc_auc_score(norm_labels, norm_probs)),
}
print(f"Color norm only — accuracy={color_norm_results['accuracy']:.4f}  "
      f"f1={color_norm_results['f1_acne']:.4f}  auroc={color_norm_results['auroc']:.4f}")

## 6. Two-Stage Few-Shot Fine-Tuning

Uses only **20 labeled DermNet samples** (10 acne + 10 non-acne) with two LR stages:

- **Stage 1 (50 epochs):** Freeze EfficientNet backbone, fine-tune classifier head only (lr=5e-3)
- **Stage 2 (30 epochs):** Unfreeze last feature block, joint fine-tuning at 100× lower backbone LR (5e-6 vs 5e-5)

All samples preprocessed with Reinhard + histogram matching before fine-tuning.

In [ ]:
# Sample 10 acne + 10 non-acne from DermNet training set
train_acne = [
    (p, 1) for p in sorted((DERMNET_DIR / 'train' / ACNE_FOLDER).glob('*'))
    if p.suffix.lower() in ('.jpg', '.jpeg', '.png')
]
train_nonacne = [
    (p, 0)
    for folder in sorted((DERMNET_DIR / 'train').iterdir())
    if folder.name != ACNE_FOLDER
    for p in sorted(folder.glob('*'))
    if p.suffix.lower() in ('.jpg', '.jpeg', '.png')
]

random.seed(42)
ft_samples = random.sample(train_acne, 10) + random.sample(train_nonacne, 10)
print(f'Few-shot set: {len(ft_samples)} samples (10 acne + 10 non-acne)')

In [ ]:
ft_train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

ft_ds     = PreprocessedDataset(ft_samples, preprocess_dermnet, ft_train_tf)
ft_loader = DataLoader(ft_ds, batch_size=10, shuffle=True, num_workers=0)

# Reload best.pth (start from clean ACNE04 checkpoint)
model = load_efficientnet(CLF_OUT / 'best.pth', device)
model.train()

# Freeze backbone
for param in model.features.parameters():
    param.requires_grad = False

criterion    = nn.CrossEntropyLoss()
optimizer_s1 = torch.optim.Adam(model.classifier.parameters(), lr=5e-3, weight_decay=1e-4)

print('Stage 1: head-only fine-tuning (50 epochs)...')
for epoch in range(50):
    for imgs, labels in ft_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer_s1.zero_grad()
        criterion(model(imgs), labels).backward()
        optimizer_s1.step()
    if (epoch + 1) % 10 == 0:
        print(f'  Stage 1 epoch {epoch+1}/50')
print('Stage 1 complete.')

In [ ]:
# Unfreeze last feature block
for param in model.features[-1].parameters():
    param.requires_grad = True

optimizer_s2 = torch.optim.Adam([
    {'params': model.features[-1].parameters(), 'lr': 5e-6},
    {'params': model.classifier.parameters(),   'lr': 5e-5},
], weight_decay=1e-4)

print('Stage 2: last block + head (30 epochs)...')
model.train()
for epoch in range(30):
    for imgs, labels in ft_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer_s2.zero_grad()
        criterion(model(imgs), labels).backward()
        optimizer_s2.step()
    if (epoch + 1) % 10 == 0:
        print(f'  Stage 2 epoch {epoch+1}/30')

torch.save(model.state_dict(), CLF_OUT / 'finetuned.pth')
print('Saved -> outputs/classifier/finetuned.pth')

## 7. Threshold Optimization

Sweep thresholds 0.05–0.95 on the same 20 few-shot samples to find the F1-maximizing decision boundary.

In [ ]:
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
thresh_ds     = PreprocessedDataset(ft_samples, preprocess_dermnet, eval_tf)
thresh_loader = DataLoader(thresh_ds, batch_size=20, shuffle=False, num_workers=0)

model.eval()
t_probs, t_labels = [], []
with torch.no_grad():
    for imgs, labels in thresh_loader:
        probs = torch.softmax(model(imgs.to(device)), dim=1)[:, 1]
        t_probs.extend(probs.cpu().numpy())
        t_labels.extend(labels.numpy())

t_probs, t_labels = np.array(t_probs), np.array(t_labels)

best_thresh, best_f1 = 0.5, 0.0
for thresh in np.arange(0.05, 0.96, 0.05):
    preds = (t_probs >= thresh).astype(int)
    f1 = f1_score(t_labels, preds, pos_label=1, zero_division=0)
    if f1 > best_f1:
        best_f1, best_thresh = f1, float(thresh)

CONF = best_thresh
print(f'Best threshold: {CONF:.2f}  (F1={best_f1:.4f} on 20 training samples)')

## 8. 6-Crop Test-Time Augmentation

FiveCrop(224) + horizontal flip of the center crop = 6 forward passes per image, averaged.

In [ ]:
_norm_tf = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
_to_t   = transforms.ToTensor()

class TTADataset(Dataset):
    def __init__(self, samples, preprocess_fn):
        self.samples, self.preprocess_fn = samples, preprocess_fn
        self._five = transforms.Compose([transforms.Resize(256), transforms.FiveCrop(224)])
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img   = self.preprocess_fn(Image.open(path).convert('RGB'))
        crops = self._five(img)                         # tuple of 5 PIL images
        center = crops[4]                               # center crop
        flip   = transforms.functional.hflip(center)
        six = torch.stack([_norm_tf(_to_t(c)) for c in list(crops) + [flip]])  # (6, 3, 224, 224)
        return six, label

tta_ds     = TTADataset(test_ds.samples, preprocess_dermnet)
tta_loader = DataLoader(tta_ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

model.eval()
ft_probs, ft_labels = [], []
with torch.no_grad():
    for crops_batch, labels in tta_loader:
        B, N, C, H, W = crops_batch.shape
        flat  = crops_batch.view(B * N, C, H, W).to(device)
        p_avg = torch.softmax(model(flat), dim=1)[:, 1].view(B, N).mean(dim=1)
        ft_probs.extend(p_avg.cpu().numpy())
        ft_labels.extend(labels.numpy())

ft_probs  = np.array(ft_probs)
ft_labels = np.array(ft_labels)
ft_preds  = (ft_probs >= CONF).astype(int)

full_results = {
    'accuracy':  float(accuracy_score(ft_labels, ft_preds)),
    'f1_acne':   float(f1_score(ft_labels, ft_preds, pos_label=1, zero_division=0)),
    'auroc':     float(roc_auc_score(ft_labels, ft_probs)),
    'threshold': float(CONF),
}
print(f"Full pipeline (color norm + 2-stage FT + thresh={CONF:.2f} + TTA)")
print(f"  accuracy={full_results['accuracy']:.4f}  "
      f"f1={full_results['f1_acne']:.4f}  auroc={full_results['auroc']:.4f}")

## 9. FaceNet VGGFace2 (Alternative Backbone)

`facenet-pytorch` InceptionResnetV1 pretrained on VGGFace2. Face-specific pretraining gives more transferable skin features than ImageNet.

- Backbone frozen, 2-class head fine-tuned on the same 20 samples
- Input size 160×160 (FaceNet standard); normalized to [−1, 1]
- Threshold optimized on the 20 samples

In [ ]:
try:
    from facenet_pytorch import InceptionResnetV1
except ImportError:
    raise ImportError("Install facenet-pytorch: pip install facenet-pytorch")

# Build model: frozen backbone + trainable 2-class head
facenet_base = InceptionResnetV1(pretrained='vggface2', classify=False).to(device)
for param in facenet_base.parameters():
    param.requires_grad = False

face_head  = nn.Linear(512, 2).to(device)
face_model = nn.Sequential(facenet_base, face_head)

# Few-shot fine-tuning (head only)
face_train_tf = transforms.Compose([
    transforms.RandomResizedCrop(160, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
face_eval_tf = transforms.Compose([
    transforms.Resize(160), transforms.CenterCrop(160),
    transforms.ToTensor(), transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

face_ft_ds     = PreprocessedDataset(ft_samples, preprocess_dermnet, face_train_tf)
face_ft_loader = DataLoader(face_ft_ds, batch_size=10, shuffle=True, num_workers=0)

face_opt = torch.optim.Adam(face_head.parameters(), lr=1e-3, weight_decay=1e-4)
face_crit = nn.CrossEntropyLoss()

print('FaceNet head fine-tuning (50 epochs)...')
face_model.train()
for epoch in range(50):
    for imgs, labels in face_ft_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        face_opt.zero_grad()
        face_crit(face_model(imgs), labels).backward()
        face_opt.step()
    if (epoch + 1) % 10 == 0:
        print(f'  epoch {epoch+1}/50')

# Threshold optimization
face_model.eval()
ft_ps, ft_ls = [], []
for path, label in ft_samples:
    img    = preprocess_dermnet(Image.open(path).convert('RGB'))
    tensor = face_eval_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        prob = torch.softmax(face_model(tensor), dim=1)[0, 1].item()
    ft_ps.append(prob); ft_ls.append(label)

ft_ps, ft_ls = np.array(ft_ps), np.array(ft_ls)
face_best_t, face_best_f1 = 0.5, 0.0
for t in np.arange(0.05, 0.96, 0.05):
    preds = (ft_ps >= t).astype(int)
    f1 = f1_score(ft_ls, preds, pos_label=1, zero_division=0)
    if f1 > face_best_f1:
        face_best_f1, face_best_t = f1, float(t)
FACE_CONF = face_best_t
print(f'FaceNet best threshold: {FACE_CONF:.2f}')

# Full DermNet test evaluation
face_test_ds     = PreprocessedDataset(test_ds.samples, preprocess_dermnet, face_eval_tf)
face_test_loader = DataLoader(face_test_ds, batch_size=32, shuffle=False, num_workers=0)

fp, fl = [], []
with torch.no_grad():
    for imgs, labels in face_test_loader:
        probs = torch.softmax(face_model(imgs.to(device)), dim=1)[:, 1]
        fp.extend(probs.cpu().numpy()); fl.extend(labels.numpy())

face_probs, face_labels = np.array(fp), np.array(fl)
face_preds = (face_probs >= FACE_CONF).astype(int)

facenet_results = {
    'accuracy':  float(accuracy_score(face_labels, face_preds)),
    'f1_acne':   float(f1_score(face_labels, face_preds, pos_label=1, zero_division=0)),
    'auroc':     float(roc_auc_score(face_labels, face_probs)),
    'threshold': float(FACE_CONF),
}
print(f"FaceNet — accuracy={facenet_results['accuracy']:.4f}  "
      f"f1={facenet_results['f1_acne']:.4f}  auroc={facenet_results['auroc']:.4f}  "
      f"thresh={facenet_results['threshold']:.2f}")

## 10. Full Ablation Table + Save Results

In [ ]:
print('\n=== Domain Adaptation Ablation ===')
print(f"{'Experiment':<28} {'Accuracy':>10} {'F1 (acne)':>10} {'AUROC':>8}")
print('-' * 60)
print(f"{'Naive baseline':<28} {naive_acc:>10.4f} {'0.0000':>10} {'—':>8}")
print(f"{'No adaptation':<28} {baseline_results['accuracy']:>10.4f} {baseline_results['f1_acne']:>10.4f} {baseline_results['auroc']:>8.4f}")
print(f"{'Color norm only':<28} {color_norm_results['accuracy']:>10.4f} {color_norm_results['f1_acne']:>10.4f} {color_norm_results['auroc']:>8.4f}")
print(f"{'Full pipeline':<28} {full_results['accuracy']:>10.4f} {full_results['f1_acne']:>10.4f} {full_results['auroc']:>8.4f}")
print(f"{'FaceNet VGGFace2':<28} {facenet_results['accuracy']:>10.4f} {facenet_results['f1_acne']:>10.4f} {facenet_results['auroc']:>8.4f}")
print(f"\nFull pipeline threshold={full_results['threshold']:.2f}  "      f"FaceNet threshold={facenet_results['threshold']:.2f}")

results_json = {
    'naive_baseline':    {'accuracy': naive_acc},
    'baseline':          baseline_results,
    'color_norm_only':   color_norm_results,
    'efficientnet_full': full_results,
    'facenet':           facenet_results,
}
out_path = OUT_PART2 / 'dermnet_results.json'
with open(out_path, 'w') as fp:
    json.dump(results_json, fp, indent=2)
print(f'\nSaved -> {out_path}')

## 11. Baseline vs Full Pipeline — Side-by-Side

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

def plot_cm(ax, labels, preds, title):
    cm = confusion_matrix(labels, preds)
    cm_display = cm.T[::-1, ::-1]
    disp = ConfusionMatrixDisplay(cm_display, display_labels=['acne', 'non-acne'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_xlabel('Actually'); ax.set_ylabel('Predicted')
    ax.set_title(title)

def plot_roc(ax, labels, probs, auroc, title):
    fpr, tpr, _ = roc_curve(labels, probs)
    ax.plot(fpr, tpr, label=f'AUROC = {auroc:.4f}')
    ax.plot([0, 1], [0, 1], 'k--', label='Random')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(title); ax.legend()

baseline_preds = (all_probs >= 0.5).astype(int)
plot_cm( axes[0, 0], all_labels, baseline_preds,
         'Confusion Matrix — Baseline (no adapt)')
plot_roc(axes[1, 0], all_labels, all_probs, baseline_results['auroc'],
         'ROC Curve — Baseline')
plot_cm( axes[0, 1], ft_labels,  ft_preds,
         f"Confusion Matrix — Full Pipeline (thresh={CONF:.2f})")
plot_roc(axes[1, 1], ft_labels,  ft_probs, full_results['auroc'],
         'ROC Curve — Full Pipeline')

plt.suptitle('Baseline vs Full Pipeline — DermNet Test Set', fontsize=14)
plt.tight_layout()
plt.savefig(OUT_PART2 / 'final_model_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> outputs/part2/final_model_results.png')